# MimicMotion [ICML 2025] — Notebook Oficial

**MimicMotion: High-Quality Human Motion Video Generation with Confidence-aware Pose Guidance**  
Yuang Zhang et al. — Tencent / Shanghai Jiao Tong University

### Highlights
- **Pose guidance con confianza** → suavidad temporal y robustez
- **Regional loss amplification** → reduce distorsión de imagen
- **Progressive latent fusion** → videos de longitud arbitraria

### Capacidades
| Modelo | Max frames | Resolución | VRAM mínima |
|--------|-----------|------------|-------------|
| MimicMotion 1.1 | 72 | 576×1024 | 16 GB (U-Net 8GB, VAE 16GB) |

> **GPU requerida**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU**  
> **Tiempo estimado**: ~20 min en 4090 para video de 35s (72 frames)

## Paso 0 — Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('GPU no detectada. Ve a: Entorno de ejecucion → Cambiar tipo → T4 GPU')

## Paso 1 — Clonar MimicMotion

Equivalente al `git clone` del README oficial.

In [ ]:
import os, subprocess

if not os.path.exists('/content/MimicMotion'):
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/tencent/MimicMotion.git',
         '/content/MimicMotion'],
        check=True
    )
    print('Repositorio clonado.')
else:
    print('Repositorio ya existe.')

os.chdir('/content/MimicMotion')
print('Directorio:', os.getcwd())

## Paso 2 — Instalar dependencias

El README recomienda `conda env create -f environment.yaml`.  
En Colab instalamos equivalente via pip con versiones compatibles con Python 3.12.

In [ ]:
import sys, subprocess

# Verificar NumPy — torch 2.x requiere NumPy <2
r = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                   capture_output=True, text=True)
ver = r.stdout.strip()
if ver and int(ver.split('.')[0]) >= 2:
    print(f'Bajando NumPy {ver} -> 1.26.4 ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==1.26.4', '--force-reinstall'], check=True)
    print('NumPy 1.26.4 instalado. Reiniciando kernel...')
    import os; os.kill(os.getpid(), 9)
else:
    print(f'NumPy {ver} OK.')

In [ ]:
import sys, subprocess

print('Instalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    # Versiones del environment.yaml adaptadas a Colab
    'torch>=2.0', 'torchvision',
    'diffusers==0.27.2',
    'transformers>=4.38,<5.0',
    'tokenizers>=0.19',
    'accelerate>=0.24',
    'omegaconf',
    'einops',
    'decord',
    'opencv-python-headless',
    'Pillow',
    'pyyaml',
    'imageio[ffmpeg]',
    'onnxruntime-gpu',
    'av',
    'huggingface_hub',
], check=True)
print('Dependencias instaladas.')

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

# Patch loader.py: safe_globals API cambio en PyTorch moderno
loader = '/content/MimicMotion/mimicmotion/utils/loader.py'
if os.path.exists(loader):
    with open(loader) as f: content = f.read()
    if 'safe_globals(*allowed_modules)' in content:
        content = content.replace('safe_globals(*allowed_modules)',
                                  'safe_globals(allowed_modules)')
        with open(loader, 'w') as f: f.write(content)
        print('Patch loader.py aplicado.')

# Patch huggingface_hub.cached_download (eliminado en hfhub>=0.24)
import huggingface_hub as _h
if not hasattr(_h, 'cached_download'):
    _h.cached_download = _h.hf_hub_download
    print('Patch cached_download aplicado.')

# Patch accelerate.clear_device_cache
import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    def _cdc():
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _aum.clear_device_cache = _cdc
    import accelerate.utils as _au; _au.clear_device_cache = _cdc
    print('Patch clear_device_cache aplicado.')

## Paso 3 — Descargar pesos

Siguiendo exactamente los comandos `wget` del README oficial.

> DWPose (~200 MB) + MimicMotion_1-1.pth (~3 GB). Total ~3.2 GB.  
> El modelo SVD se descarga automáticamente durante la inferencia (requiere HF_TOKEN).

In [ ]:
import os

os.chdir('/content/MimicMotion')
os.makedirs('models/DWPose', exist_ok=True)

# 1. DWPose — comandos wget del README
dwpose_files = [
    ('https://huggingface.co/yzd-v/DWPose/resolve/main/yolox_l.onnx',
     'models/DWPose/yolox_l.onnx'),
    ('https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.onnx',
     'models/DWPose/dw-ll_ucoco_384.onnx'),
]
for url, dst in dwpose_files:
    if not os.path.exists(dst):
        print(f'Descargando {dst}...')
        subprocess.run(['wget', '-q', url, '-O', dst], check=True)
    else:
        print(f'{dst} ya existe.')

# 2. MimicMotion 1.1 checkpoint — wget -P del README
if not os.path.exists('models/MimicMotion_1-1.pth'):
    print('Descargando MimicMotion_1-1.pth (~3 GB)...')
    subprocess.run([
        'wget', '-q',
        'https://huggingface.co/tencent/MimicMotion/resolve/main/MimicMotion_1-1.pth',
        '-P', 'models/'
    ], check=True)
    print('MimicMotion_1-1.pth descargado.')
else:
    print('MimicMotion_1-1.pth ya existe.')

# Verificar estructura esperada por el README
print('\nEstructura de modelos:')
for root, dirs, files in os.walk('models'):
    level = root.replace('models', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1e6
        print(f'{indent}  {f} ({size:.0f} MB)')

## Paso 4 — Token HuggingFace (para SVD)

El SVD se descarga automáticamente durante la inferencia.  
Necesitas aceptar los términos en [stabilityai/stable-video-diffusion-img2vid-xt-1-1](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1).

In [ ]:
from huggingface_hub import login, whoami

HF_TOKEN = ''  # @param {type:"string"}
# Alternativa: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    try:
        print(f'Autenticado como: {whoami()["name"]}')
    except Exception as e:
        print(f'Token aceptado: {e}')
else:
    print('Sin token — SVD no podra descargarse automáticamente.')

## Paso 5 — Subir archivos de entrada

In [ ]:
from google.colab import files
import os

os.makedirs('/content/inputs', exist_ok=True)

print('Sube la FOTO del personaje (jpg/png):')
uploaded = files.upload()
ref_image_path = None
for fname, data in uploaded.items():
    ref_image_path = f'/content/inputs/{fname}'
    with open(ref_image_path, 'wb') as f: f.write(data)
    print(f'Foto: {ref_image_path}')

In [ ]:
print('Sube el VIDEO DE REFERENCIA (mp4):')
uploaded = files.upload()
ref_video_path = None
for fname, data in uploaded.items():
    ref_video_path = f'/content/inputs/{fname}'
    with open(ref_video_path, 'wb') as f: f.write(data)
    print(f'Video: {ref_video_path}')

In [ ]:
from PIL import Image
import IPython.display as ipd, cv2

assert ref_image_path and os.path.exists(ref_image_path), 'Foto no encontrada.'
assert ref_video_path and os.path.exists(ref_video_path), 'Video no encontrado.'

img = Image.open(ref_image_path)
print(f'Foto: {img.size[0]}x{img.size[1]} px')
ipd.display(img.resize((256, int(256 * img.size[1] / img.size[0]))))

cap = cv2.VideoCapture(ref_video_path)
nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {w}x{h} | {nf} frames | {fps:.1f} fps | {nf/fps:.1f}s')

## Paso 6 — Configurar inferencia (test.yaml)

El README usa `configs/test.yaml`. Aquí lo generamos dinámicamente.  
Referencia de parámetros del README oficial:

| Parámetro | Descripción | Recomendado |
|---|---|---|
| `resolution` | Alto en píxeles | 576 (nativo) |
| `num_frames` | Frames por tile | 16–72 |
| `sample_stride` | Intervalo de muestreo | 2–4 |
| `frames_overlap` | Solapamiento entre tiles | 6 |
| `num_inference_steps` | Pasos de difusión | 25 |
| `guidance_scale` | Fuerza de seguimiento de pose | 2.0 |

> **VRAM:** U-Net mínimo 8 GB. VAE mínimo 16 GB (o ejecutar en CPU).

In [ ]:
import yaml, os

# Parámetros — equivalentes a configs/test.yaml del README
RESOLUTION          = 576   # @param {type:"slider", min:256, max:576, step:64}
NUM_FRAMES          = 16    # @param {type:"slider", min:8, max:72, step:8}
SAMPLE_STRIDE       = 2     # @param {type:"slider", min:1, max:4, step:1}
FRAMES_OVERLAP      = 6     # @param {type:"slider", min:2, max:16, step:2}
NUM_INFERENCE_STEPS = 25    # @param {type:"slider", min:10, max:50, step:5}
NOISE_AUG_STRENGTH  = 0.0   # @param {type:"number"}
GUIDANCE_SCALE      = 2.0   # @param {type:"number"}
OUTPUT_FPS          = 15    # @param {type:"slider", min:8, max:30, step:1}
SEED                = 42    # @param {type:"integer"}

config = {
    'base_model_path': 'stabilityai/stable-video-diffusion-img2vid-xt-1-1',
    'ckpt_path': 'models/MimicMotion_1-1.pth',
    'test_cases': [{
        'ref_video_path': ref_video_path,
        'ref_image_path': ref_image_path,
        'num_frames':          NUM_FRAMES,
        'resolution':          RESOLUTION,
        'frames_overlap':      FRAMES_OVERLAP,
        'num_inference_steps': NUM_INFERENCE_STEPS,
        'noise_aug_strength':  NOISE_AUG_STRENGTH,
        'guidance_scale':      GUIDANCE_SCALE,
        'sample_stride':       SAMPLE_STRIDE,
        'fps':                 OUTPUT_FPS,
        'seed':                SEED,
    }]
}

os.makedirs('configs', exist_ok=True)
config_path = 'configs/colab_test.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config guardada en: {config_path}')
print(yaml.dump(config, default_flow_style=False))

## Paso 7 — Ejecutar inferencia

Equivalente al comando oficial del README:
```bash
python inference.py --inference_config configs/test.yaml
```
Con la variable de entorno recomendada para VRAM limitada:
```bash
PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:256
```

In [ ]:
import os, sys

MIMIC_DIR = '/content/MimicMotion'
os.chdir(MIMIC_DIR)
os.makedirs('outputs', exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:256'

if MIMIC_DIR not in sys.path:
    sys.path.insert(0, MIMIC_DIR)

# Wrapper: aplica patches y ejecuta inference.py en el mismo proceso
import huggingface_hub as _h
if not hasattr(_h, 'cached_download'):
    _h.cached_download = _h.hf_hub_download

import torch
import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    def _cdc():
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _aum.clear_device_cache = _cdc
    import accelerate.utils as _au; _au.clear_device_cache = _cdc

# Ejecutar inference.py — replica exacta de:
# python inference.py --inference_config configs/colab_test.yaml --output_dir outputs/
sys.argv = ['inference.py',
            '--inference_config', config_path,
            '--output_dir', 'outputs/']

print('Ejecutando: python inference.py --inference_config', config_path)
print('-' * 60)

with open(os.path.join(MIMIC_DIR, 'inference.py')) as _f:
    exec(_f.read(), {'__name__': '__main__',
                     '__file__': os.path.join(MIMIC_DIR, 'inference.py')})

## Paso 8 — Ver y descargar el resultado

In [ ]:
import glob, os
from IPython.display import HTML, display as ipy_display
from base64 import b64encode

videos = sorted(
    glob.glob('/content/MimicMotion/outputs/**/*.mp4', recursive=True),
    key=os.path.getmtime, reverse=True
)

if not videos:
    print('No se encontraron videos. Revisa los errores del paso anterior.')
else:
    out = videos[0]
    print(f'Video: {out} ({os.path.getsize(out)/1e6:.1f} MB)')
    with open(out, 'rb') as f:
        b64 = b64encode(f.read()).decode()
    ipy_display(HTML(
        f'<video controls width="540" autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))

In [ ]:
from google.colab import files
if videos:
    print(f'Descargando {os.path.basename(out)}...')
    files.download(out)
else:
    print('No hay video para descargar.')

## (Avanzado) Generar video largo con Progressive Latent Fusion

El README indica que con `frames_overlap` adecuado se pueden generar videos de longitud arbitraria.  
Incrementa `NUM_FRAMES` hasta 72 y usa `FRAMES_OVERLAP=6` para suavizar transiciones.

**Requerimientos de VRAM según el README:**
```
72 frames → 16 GB VRAM → ~20 min en 4090
16 frames → 8 GB VRAM (U-Net) + 16 GB (VAE, puede correr en CPU)
```

In [ ]:
import yaml, os

# Configuracion para video largo (72 frames max, resolution 576)
LONG_NUM_FRAMES    = 72   # @param {type:"slider", min:16, max:72, step:8}
LONG_FRAMES_OVERLAP = 6   # @param {type:"slider", min:2, max:16, step:2}
LONG_SAMPLE_STRIDE  = 2   # @param {type:"slider", min:1, max:4, step:1}

config_long = {
    'base_model_path': 'stabilityai/stable-video-diffusion-img2vid-xt-1-1',
    'ckpt_path': 'models/MimicMotion_1-1.pth',
    'test_cases': [{
        'ref_video_path': ref_video_path,
        'ref_image_path': ref_image_path,
        'num_frames':          LONG_NUM_FRAMES,
        'resolution':          576,
        'frames_overlap':      LONG_FRAMES_OVERLAP,
        'num_inference_steps': 25,
        'noise_aug_strength':  0.0,
        'guidance_scale':      2.0,
        'sample_stride':       LONG_SAMPLE_STRIDE,
        'fps':                 15,
        'seed':                42,
    }]
}

config_long_path = 'configs/colab_long.yaml'
with open(config_long_path, 'w') as f:
    yaml.dump(config_long, f, default_flow_style=False)

print(f'Config para video largo: {config_long_path}')
print(f'Frames: {LONG_NUM_FRAMES} | Overlap: {LONG_FRAMES_OVERLAP} | Stride: {LONG_SAMPLE_STRIDE}')
print('\nEjecuta la siguiente celda cuando estes listo.')

In [ ]:
sys.argv = ['inference.py',
            '--inference_config', config_long_path,
            '--output_dir', 'outputs/long/']

os.makedirs('outputs/long', exist_ok=True)
print(f'Ejecutando con {LONG_NUM_FRAMES} frames...')
print('-' * 60)

with open(os.path.join(MIMIC_DIR, 'inference.py')) as _f:
    exec(_f.read(), {'__name__': '__main__',
                     '__file__': os.path.join(MIMIC_DIR, 'inference.py')})

---
## Referencia — README oficial MimicMotion

```bash
# Comandos originales del README
conda env create -f environment.yaml
conda activate mimicmotion

mkdir -p models/DWPose
wget https://huggingface.co/yzd-v/DWPose/resolve/main/yolox_l.onnx -O models/DWPose/yolox_l.onnx
wget https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.onnx -O models/DWPose/dw-ll_ucoco_384.onnx
wget -P models/ https://huggingface.co/tencent/MimicMotion/resolve/main/MimicMotion_1-1.pth

python inference.py --inference_config configs/test.yaml
```

```
models/
├── DWPose
│   ├── dw-ll_ucoco_384.onnx
│   └── yolox_l.onnx
└── MimicMotion_1-1.pth
```

**Paper:** MimicMotion: High-Quality Human Motion Video Generation with Confidence-aware Pose Guidance  
**Aceptado:** ICML 2025  
**Autores:** Yuang Zhang, Jiaxi Gu, Li-Wen Wang, Han Wang, Junqi Cheng, Yuefeng Zhu, Fangyuan Zou  
**Instituciones:** Tencent / Shanghai Jiao Tong University